In [1]:
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt
import importlib.util

In [ ]:
#2. Isolation Forest
chemin_isolation_forest = r"Ecrire le bon chemin"

if os.path.exists(chemin_isolation_forest):
    print(f"Fichier trouvé: {chemin_isolation_forest}")
else:
    print(f"Fichier NON trouvé: {chemin_isolation_forest}")
    exit(1)

spec = importlib.util.spec_from_file_location("isolation_forest", chemin_isolation_forest)
isolation_forest = importlib.util.module_from_spec(spec)
spec.loader.exec_module(isolation_forest)

IsolationForest = isolation_forest.IsolationForest

Fichier trouvé: D:\COURS IFRI\L2\Semestre 4\Concept et Application de l'Apprentissage Automatique\ifri_mini_ml_lib\ifri_mini_ml_lib\anomalies_detection\isolation_forest.py


In [ ]:
chemin_dataset = r"Ecrire le bon chemin"

if os.path.exists(chemin_dataset):
    print(f"Dataset trouvé: {chemin_dataset}")
else:
    print(f"Dataset NON trouvé: {chemin_dataset}")
    exit(1)

df = pd.read_csv(chemin_dataset)

print(f"Nombre de transactions: {df.shape[0]}")
print(f"Nombre de colonnes: {df.shape[1]}")
print(df.columns.tolist())
print(df[['Time', 'Amount']].describe())
print(f"Nombre de fraudes (Class=1): {df['Class'].sum()}")
print(f"Nombre de transactions normales: {(df['Class']==0).sum()}")

Dataset trouvé: D:\COURS IFRI\L2\Semestre 4\Concept et Application de l'Apprentissage Automatique\TP\TP_Anomalies\creditcard.csv
Nombre de transactions: 284807
Nombre de colonnes: 31
['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']
                Time         Amount
count  284807.000000  284807.000000
mean    94813.859575      88.349619
std     47488.145955     250.120109
min         0.000000       0.000000
25%     54201.500000       5.600000
50%     84692.000000      22.000000
75%    139320.500000      77.165000
max    172792.000000   25691.160000
Nombre de fraudes (Class=1): 492
Nombre de transactions normales: 284315


In [4]:
# Travailler uniquement sur la colonne Amount
X = df[['Amount']].values
y = df['Class'].values

print(f"Shape de X: {X.shape}")

Shape de X: (284807, 1)


In [ ]:
model = IsolationForest(n_trees=100, sample_size=256, contamination=0.05)
model.fit(X)
print(f"Fit terminé")

scores = model.anomaly_score(X)
print(f"Anomaly score terminé")

labels = (scores >= model.threshold).astype(int)

Fit terminé en 173.1s
Anomaly score terminé en 117.4s


In [6]:
print(f"Anomalies détectées: {labels.sum()}")
print(f"Taux d'anomalies: {labels.sum() / len(labels) * 100:.2f}%")

Anomalies détectées: 14302
Taux d'anomalies: 5.02%


In [7]:
# Comparaison avec les vraies fraudes
y = df['Class'].values
true_positives = ((labels == 1) & (y == 1)).sum()
false_positives = ((labels == 1) & (y == 0)).sum()
false_negatives = ((labels == 0) & (y == 1)).sum()

print(f"Vraies fraudes détectées (True Positives): {true_positives} / {y.sum()}")
print(f"Fausses alertes (False Positives): {false_positives}")
print(f"Fraudes manquées (False Negatives): {false_negatives}")

Vraies fraudes détectées (True Positives): 44 / 492
Fausses alertes (False Positives): 14258
Fraudes manquées (False Negatives): 448


## Difficultés et observations

L'implémentation d'Isolation Forest, codée en Python pur sans vectorisation, 
s'est révélée très lente sur les 284 807 transactions : Le temps d'exécution varie légèrement entre les exécutions (~270-290s) en raison de la nature aléatoire de l'algorithme.

Sur le plan des résultats, en utilisant uniquement la colonne Amount, le modèle 
détecte seulement 44 fraudes sur 492 (9%), avec un taux élevé de faux positifs 
(14 258). Cela montre que le montant seul n'est pas suffisamment discriminant 
pour distinguer une transaction frauduleuse d'une transaction normale, les 
informations les plus pertinentes se trouvant probablement dans les variables 
anonymisées V1 à V28.